!Working with local and remote datasets:
Data format	Loading script	Example

CSV & TSV	|| csv	|| # load_dataset("csv", data_files="my_file.csv")

Text files	||  text	||  # load_dataset("text", data_files="my_file.txt")

JSON & JSON Lines||  	json	|| # load_dataset("json", data_files="my_file.jsonl")

Pickled DataFrames	||  pandas	||  # load_dataset("pandas", data_files="my_dataframe.pkl")

In [ ]:
!pip install -U transformers

# local loading dataset

use the SQuAD-it dataset

In [ ]:
!wget https://github.com/crux82/squad-it/raw/master/SQuAD_it-train.json.gz
!wget https://github.com/crux82/squad-it/raw/master/SQuAD_it-test.json.gz

In [ ]:
# !gzip -dkv SQuAD_it-*.json.gz


In [ ]:
from datasets import load_dataset


# squad_it_dataset = load_dataset("json", data_files="SQuAD_it-train.json", field="data")


data_files = {"train": "SQuAD_it-train.json.gz", "test": "SQuAD_it-test.json.gz"}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
squad_it_dataset

In [ ]:
squad_it_dataset["train"][0]

apply Dataset.map()

In [ ]:
data_files = {"train": "/content/SQuAD_it-train.json.gz", "test": "/content/SQuAD_it-train.json.gz"}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
squad_it_dataset

Loading a remote dataset

In [ ]:
url = "https://github.com/crux82/squad-it/raw/master/"
data_files = {
    "train": url + "SQuAD_it-train.json.gz",
    "test": url + "SQuAD_it-test.json.gz",
}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")


In [ ]:
squad_it_dataset

# Slicing and dicing our data

In [ ]:
!wget http://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
!unzip drugsCom_raw.zip

In [ ]:
from datasets import load_dataset

data_files = {"train": "/content/drugsComTrain_raw.tsv", "test": "/content/drugsComTest_raw.tsv"}
# \t is the tab character in Python
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")

drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)

lambda {arguments : expression}

In [ ]:
drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)



drug_dataset = drug_dataset.map(lowercase_condition)
# Check that lowercasing worked
drug_dataset["train"]["condition"][:3]

# Creating new columns

In [ ]:
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}

drug_dataset = drug_dataset.map(compute_review_length)
# Inspect the first training example
drug_dataset["train"][0]

We can sort this new column with Dataset.sort() to see what the extreme values look like:

In [ ]:
drug_dataset["train"].sort("review_length")[:3]

An alternative way to add new columns to a dataset is with the Dataset.add_column() function. This allows you to provide the column as a Python list or NumPy array and can be handy in situations where Dataset.map() is not well suited for your analysis.

Let’s use the Dataset.filter() function to remove reviews that contain fewer than 30 words. Similarly to what we did with the condition column, we can filter out the very short reviews by requiring that the reviews have a length above this threshold

In [ ]:
drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
print(drug_dataset.num_rows)

The last thing we need to deal with is the presence of HTML character codes , use Dataset.map() to unescape all the HTML characters in our corpus:

In [ ]:
import html
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

# The map() method's superpowers

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["review"], truncation=True)

In [ ]:
# batched=True, num_proc speed up

slow_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)


def slow_tokenize_function(examples):
    return slow_tokenizer(examples["review"], truncation=True)


tokenized_dataset = drug_dataset.map(slow_tokenize_function, batched=True, num_proc=8)

return_overflowing_tokens=True

In [ ]:
def tokenize_and_split(examples):
    result = tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )
    # Extract mapping between new and old indices
    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result
tokenized_dataset = drug_dataset.map(
    tokenize_and_split, batched=True, remove_columns=drug_dataset["train"].column_names
)

From Dataset s to DataFrame s and back

In [ ]:
drug_dataset.set_format("pandas")
drug_dataset["train"][:3]


In [ ]:
# create a pandas.DataFrame for the whole training set by selecting all the elements
train_df = drug_dataset["train"][:]

# Creating a validation set

In [ ]:
drug_dataset_clean = drug_dataset["train"].train_test_split(train_size=0.8, seed=42)
# Rename the default "test" split to "validation"
drug_dataset_clean["validation"] = drug_dataset_clean.pop("test")
# Add the "test" set to our `DatasetDict`
drug_dataset_clean["test"] = drug_dataset["test"]
drug_dataset_clean

# Saving a dataset
Data format           	Function

Arrow:->                Dataset.save_to_disk()

CSV:	   ->            Dataset.to_csv()

JSON:	    ->         Dataset.to_json()

In [ ]:
drug_dataset_clean.save_to_disk("drug-reviews")